In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — INSTALL & IMPORTS
# ════════════════════════════════════════════════════════════
!pip install gspread gspread-dataframe openpyxl -q

import pandas as pd
import numpy as np
import warnings
import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import auth, files
from google.auth import default
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
warnings.filterwarnings('ignore')
print('✅ Libraries ready')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 2 — CONFIG  (edit these values)
# ════════════════════════════════════════════════════════════

CONFIG = {
    # ── Tickers ─────────────────────────────────────────────
    'TICKERS': ['AMZN', 'GOOGL', 'MSFT', 'ORCL'],

    # ── Valuation metrics ────────────────────────────────────
    'METRICS': ['pe_fy1', 'ev_rev_fy1', 'ev_ebitda_fy1', 'ev_fcf_ttm'],

    # ── Rolling signal window ────────────────────────────────
    'SIGNAL_WINDOW': 252,
    'MIN_OBS':        20,

    # ── 5-bucket thresholds (D1/Q2/Q3/Q4/D5) ────────────────
    'BUCKET_CUTS': [0, 0.10, 0.25, 0.75, 0.90, 1.01],
    'BUCKET_KEYS': ['D1_cheap','Q2_cheap_mid','Q3_neutral','Q4_rich_mid','D5_expensive'],
    'BUCKET_NAMES':['D1 Cheap (bot 10%)','Q2 Cheap-Mid (10-25%)','Q3 Neutral (25-75%)',
                    'Q4 Rich-Mid (75-90%)','D5 Expensive (top 10%)'],

    # ── Forward return horizons (trading days) ───────────────
    'HORIZONS': [30, 60, 90, 120],

    # ── Shock calibration ────────────────────────────────────
    'SHOCK_WINDOWS':    [3, 5],
    'POST_WINDOWS':     [3, 5],
    'SHOCK_THRESHOLDS': [0.03, 0.05],

    # ── Google Sheet URL  (paste yours here) ─────────────────
    'SHEET_URL': 'https://docs.google.com/spreadsheets/d/YOUR_SHEET_ID_HERE/edit',

    # ── Output Excel filename ────────────────────────────────
    'OUTPUT_EXCEL': 'Valuation_Anchor_v1.xlsx',

    # ── Source file name (uploaded below) ───────────────────
    'SOURCE_FILE': 'valuation_data.xlsx',
}

METRIC_LABELS = {
    'pe_fy1':'P/E FY1', 'ev_rev_fy1':'EV/Rev FY1',
    'ev_ebitda_fy1':'EV/EBITDA FY1', 'ev_fcf_ttm':'EV/FCF TTM'
}
print('✅ Config loaded')
print(f"  Tickers : {CONFIG['TICKERS']}")
print(f"  Metrics : {CONFIG['METRICS']}")
print(f"  Horizons: {CONFIG['HORIZONS']} trading days")
print(f"  Shocks  : {CONFIG['SHOCK_WINDOWS']}D backward, {CONFIG['POST_WINDOWS']}D post")

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 3 — UPLOAD RAW DATA FILE
# ════════════════════════════════════════════════════════════
print('Upload your raw data Excel file when the file picker appears...')
uploaded = files.upload()
fname = list(uploaded.keys())[0]
print(f'✅ Uploaded: {fname}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 4 — READ & CLEAN RAW DATA
# ════════════════════════════════════════════════════════════
def load_raw(fname, tickers):
    """Read raw_data sheet, rename meta cols, forward-fill blanks."""
    df = pd.read_excel(fname, sheet_name='raw_data', header=0)
    META = ['Symbol','Name','Metric','Series_key']
    df = df.rename(columns={df.columns[i]: META[i] for i in range(4)})

    date_cols_raw = df.columns[4:]
    date_strs = [
        c.strftime('%Y-%m-%d') if hasattr(c, 'strftime')
        else pd.to_datetime(c).strftime('%Y-%m-%d')
        for c in date_cols_raw
    ]
    df = df.rename(columns=dict(zip(date_cols_raw, date_strs)))
    date_cols = date_strs

    df_co = df[df['Symbol'].isin(tickers)].copy()
    df_co[date_cols] = df_co[date_cols].apply(pd.to_numeric, errors='coerce')

    # Forward-fill only (bfill on wide row = carry last known value forward in time)
    df_co[date_cols] = df_co[date_cols].apply(lambda row: row.bfill(), axis=1)
    blanks = df_co[date_cols].isna().sum().sum()
    print(f'  Blanks after fill: {blanks}  (should be 0)')
    return df_co, date_cols

df_co, date_cols = load_raw(fname, CONFIG['TICKERS'])
print(f'✅ Raw data: {df_co.shape[0]} series × {len(date_cols)} dates')
print(f'   Date range: {date_cols[0]} (newest) → {date_cols[-1]} (oldest)')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 5 — BUILD panel_daily (single source of truth)
# ════════════════════════════════════════════════════════════
META = ['Symbol','Name','Metric','Series_key']

df_long = df_co.melt(id_vars=META, value_vars=date_cols, var_name='date', value_name='value')
df_long['date']  = pd.to_datetime(df_long['date'])
df_long['value'] = pd.to_numeric(df_long['value'], errors='coerce')

df_piv = df_long.pivot_table(
    index=['date','Symbol'], columns='Series_key', values='value'
).reset_index()
df_piv.columns.name = None

# ── FCF validity flags ─────────────────────────────────────
df_piv['fcf_flag'] = np.where(
    df_piv['fcf_ttm'].isna(), 'missing',
    np.where(df_piv['fcf_ttm'] < 0, 'negative_fcf', 'valid')
)

# ── Derived ratios ─────────────────────────────────────────
df_piv['pe_fy1']        = df_piv['price'] / df_piv['eps_est_fy1'].replace(0, np.nan)
df_piv['ev_rev_fy1']    = df_piv['ev']    / df_piv['revenue_est_fy1'].replace(0, np.nan)
df_piv['ev_ebitda_fy1'] = df_piv['ev']    / df_piv['ebitda_est_fy1'].replace(0, np.nan)
df_piv['ev_fcf_ttm']    = df_piv['ev']    / df_piv['fcf_ttm'].replace(0, np.nan)

# ── Log multiples ──────────────────────────────────────────
for col in CONFIG['METRICS']:
    df_piv[f'log_{col}'] = np.where(df_piv[col] > 0, np.log(df_piv[col]), np.nan)
df_piv['log_ev_fcf_flag'] = np.where(
    df_piv['ev_fcf_ttm'].isna(), 'missing',
    np.where(df_piv['ev_fcf_ttm'] <= 0, 'invalid_neg_fcf', 'valid')
)

# ── Rolling 252d z-score + percentile rank ─────────────────
WINDOW  = CONFIG['SIGNAL_WINDOW']
MIN_OBS = CONFIG['MIN_OBS']

for ticker in CONFIG['TICKERS']:
    mask = df_piv['Symbol'] == ticker
    df_t = df_piv[mask].sort_values('date')
    for mk in CONFIG['METRICS']:
        lm = f'log_{mk}'
        s   = df_t[lm]
        rm  = s.rolling(WINDOW, min_periods=MIN_OBS).mean()
        rs  = s.rolling(WINDOW, min_periods=MIN_OBS).std()
        z   = (s - rm) / rs
        pct = s.rolling(WINDOW, min_periods=MIN_OBS).apply(
            lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False
        )
        df_piv.loc[mask, f'zscore_{mk}'] = z.values
        df_piv.loc[mask, f'pct_{mk}']    = pct.values

# ── 5-bucket assignment ────────────────────────────────────
for ticker in CONFIG['TICKERS']:
    mask = df_piv['Symbol'] == ticker
    for mk in CONFIG['METRICS']:
        df_piv.loc[mask, f'bucket_{mk}'] = pd.cut(
            df_piv.loc[mask, f'pct_{mk}'],
            bins=CONFIG['BUCKET_CUTS'],
            labels=CONFIG['BUCKET_KEYS'],
            right=False
        ).astype(str).replace('nan', '')

# ── Forward returns (calendar × 1.42 to account for weekends) ─
def compute_fwd_returns(price_s, horizon_td):
    dates  = price_s.index
    prices = price_s.values
    fwd    = np.full(len(prices), np.nan)
    for i in range(len(prices)):
        target = dates[i] + pd.Timedelta(days=int(horizon_td * 1.42))
        j = np.searchsorted(dates, target)
        if j < len(prices):
            fwd[i] = (prices[j] - prices[i]) / prices[i]
    return fwd

for ticker in CONFIG['TICKERS']:
    mask = df_piv['Symbol'] == ticker
    px   = df_piv[mask].sort_values('date').set_index('date')['price']
    for h in CONFIG['HORIZONS']:
        fwd = compute_fwd_returns(px, h)
        df_piv.loc[df_piv[mask].sort_values('date').index, f'fwd_{h}d'] = fwd

df_piv = df_piv.sort_values(['Symbol','date'], ascending=[True, False])
print(f'✅ panel_daily: {df_piv.shape[0]:,} rows × {df_piv.shape[1]} cols')
print(f'   Date range: {df_piv["date"].min().date()} → {df_piv["date"].max().date()}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 6 — MODULE 1: VALUATION BUCKET TABLES
# ════════════════════════════════════════════════════════════
def dist_stats(r):
    """Full distribution stats for a return series."""
    r = r.dropna()
    if len(r) < 5: return None
    return {
        'n':        len(r),
        'median':   round(r.median() * 100, 2),
        'mean':     round(r.mean()   * 100, 2),
        'p25':      round(r.quantile(0.25) * 100, 2),
        'p75':      round(r.quantile(0.75) * 100, 2),
        'iqr':      round((r.quantile(0.75) - r.quantile(0.25)) * 100, 2),
        'p_up':     round((r > 0).mean() * 100, 1),
        'p_dn5':    round((r < -0.05).mean() * 100, 1),
        'p_dn10':   round((r < -0.10).mean() * 100, 1),
        'p_dn20':   round((r < -0.20).mean() * 100, 1),
        'p10':      round(r.quantile(0.10) * 100, 2),
        'p90':      round(r.quantile(0.90) * 100, 2),
        'skew':     round(float(r.skew()), 3),
        'std':      round(r.std() * 100, 2),
        'stability':'Stable' if len(r) >= 50 else 'Noisy',
    }

val_rows = []
for ticker in CONFIG['TICKERS']:
    df_t = df_piv[df_piv['Symbol'] == ticker].sort_values('date').copy()
    for mk in CONFIG['METRICS']:
        bkt_col = f'bucket_{mk}'
        for bkey, bname in zip(CONFIG['BUCKET_KEYS'], CONFIG['BUCKET_NAMES']):
            mask = df_t[bkt_col] == bkey
            for h in CONFIG['HORIZONS']:
                fwd = df_t.loc[mask, f'fwd_{h}d']
                st  = dist_stats(fwd)
                if st is None: continue
                row = {
                    'ticker': ticker, 'metric': METRIC_LABELS[mk],
                    'metric_key': mk, 'bucket_key': bkey,
                    'bucket_name': bname, 'horizon': h, 'horizon_lbl': f'{h}d',
                }
                row.update(st)
                val_rows.append(row)

df_val = pd.DataFrame(val_rows)
print(f'✅ Valuation bucket table: {len(df_val)} rows')
print('\n  Sample AMZN EV/Rev FY1 @ 60d:')
s = df_val[(df_val['ticker']=='AMZN') & (df_val['metric_key']=='ev_rev_fy1') & (df_val['horizon']==60)]
print(s[['bucket_name','n','median','p_up','p_dn10']].to_string(index=False))

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 7 — METRIC LEADERBOARD + HALF-LIFE
# ════════════════════════════════════════════════════════════

# ── Leaderboard ────────────────────────────────────────────
leader_rows = []
for ticker in CONFIG['TICKERS']:
    for mk in CONFIG['METRICS']:
        for h in CONFIG['HORIZONS']:
            sub   = df_val[(df_val['ticker']==ticker) & (df_val['metric_key']==mk) & (df_val['horizon']==h)]
            cheap = sub[sub['bucket_key']=='D1_cheap']
            exp   = sub[sub['bucket_key']=='D5_expensive']
            if len(cheap) == 0 or len(exp) == 0: continue
            spread    = round(float(cheap['median'].values[0]) - float(exp['median'].values[0]), 2)
            cheap_hit = float(cheap['p_up'].values[0])
            cheap_n   = int(cheap['n'].values[0])
            df_t   = df_piv[df_piv['Symbol']==ticker]
            merged = df_t[[f'pct_{mk}', f'fwd_{h}d']].dropna()
            ic = round(float(merged[f'pct_{mk}'].corr(merged[f'fwd_{h}d'], method='spearman')), 4) \
                 if len(merged) > 20 else np.nan
            leader_rows.append({
                'ticker': ticker, 'metric': METRIC_LABELS[mk], 'metric_key': mk,
                'horizon': f'{h}d', 'spread_median': spread,
                'cheap_hit_rate': cheap_hit, 'cheap_n': cheap_n,
                'ic_spearman': ic,
                'rank_quality': round(abs(spread) * cheap_hit / 100, 2),
            })

df_leader = pd.DataFrame(leader_rows)
print('✅ Leaderboard built')
print('\n  Best anchor per ticker @ 60d:')
top = (df_leader[df_leader['horizon']=='60d']
       .sort_values(['ticker','spread_median'], ascending=[True,False])
       .groupby('ticker').first().reset_index())
print(top[['ticker','metric','spread_median','cheap_hit_rate','ic_spearman']].to_string(index=False))

# ── Best anchor lookup (used in current_signals) ──────────
best_anchor = {}
for ticker in CONFIG['TICKERS']:
    sub = df_leader[(df_leader['ticker']==ticker) & (df_leader['horizon']=='60d')]
    if len(sub):
        best_anchor[ticker] = sub.sort_values('rank_quality', ascending=False).iloc[0]['metric']
print(f'\n  Best anchors (60d): {best_anchor}')

# ── Half-life ──────────────────────────────────────────────
hl_rows = []
for ticker in CONFIG['TICKERS']:
    df_t = df_piv[df_piv['Symbol']==ticker].sort_values('date')
    for mk in CONFIG['METRICS']:
        s   = df_t[f'log_{mk}'].dropna().reset_index(drop=True)
        if len(s) < 30: continue
        rm  = s.rolling(252, min_periods=20).mean()
        gap = (s - rm).dropna()
        if len(gap) < 20: continue
        y = gap.iloc[1:].values
        x = gap.iloc[:-1].values
        b = float(np.cov(x, y)[0,1] / np.var(x)) if np.var(x) > 0 else np.nan
        hl = round(-np.log(2)/np.log(b), 1) if b and 0 < b < 1 else np.nan
        speed = ('Fast (<50d)' if hl and hl < 50 else
                 'Medium (50-150d)' if hl and hl < 150 else 'Slow (>150d)')
        hl_rows.append({
            'ticker': ticker, 'metric': METRIC_LABELS[mk], 'metric_key': mk,
            'ar1_coef': round(b, 4) if b else np.nan,
            'halflife_days': hl, 'speed': speed
        })

df_hl = pd.DataFrame(hl_rows)
print('\n✅ Half-life computed')
print(df_hl[['ticker','metric','halflife_days','speed']].to_string(index=False))

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 8 — MODULE 2: FAST SHOCK CALIBRATION
# ════════════════════════════════════════════════════════════
shock_rows = []

for ticker in CONFIG['TICKERS']:
    df_t   = df_piv[df_piv['Symbol']==ticker].sort_values('date').reset_index(drop=True)
    prices = df_t['price'].values
    pct_pe = df_t[f'pct_{CONFIG["METRICS"][0]}'].values  # P/E pct for val state

    for sw in CONFIG['SHOCK_WINDOWS']:
        bwd = np.full(len(prices), np.nan)
        for i in range(sw, len(prices)):
            if prices[i-sw] > 0:
                bwd[i] = (prices[i] - prices[i-sw]) / prices[i-sw]

        for thresh in CONFIG['SHOCK_THRESHOLDS']:
            for direction in ['DOWN', 'UP']:
                idx_set = np.where(
                    (bwd <= -thresh if direction == 'DOWN' else bwd >= thresh)
                    & ~np.isnan(bwd)
                )[0]

                for pw in CONFIG['POST_WINDOWS']:
                    post_rets  = []
                    val_states = []
                    for i in idx_set:
                        j = i + pw
                        if j < len(prices) and prices[i] > 0:
                            post_rets.append((prices[j] - prices[i]) / prices[i])
                            pr = pct_pe[i]
                            val_states.append(
                                'cheap'     if (not np.isnan(pr) and pr <= 0.25) else
                                'expensive' if (not np.isnan(pr) and pr >= 0.75) else 'neutral'
                            )

                    # ALL (unconditional)
                    st = dist_stats(pd.Series(post_rets))
                    if st:
                        row = {'ticker': ticker, 'shock_window': f'{sw}D',
                               'post_window': f'{pw}D', 'direction': direction,
                               'threshold': f'{int(thresh*100)}%',
                               'val_state': 'ALL', 'n_events': len(post_rets)}
                        row.update(st); shock_rows.append(row)

                    # By valuation state
                    for vs in ['cheap', 'neutral', 'expensive']:
                        vs_r = pd.Series([post_rets[k] for k,v in enumerate(val_states) if v == vs])
                        st2  = dist_stats(vs_r)
                        if st2:
                            r2 = {'ticker': ticker, 'shock_window': f'{sw}D',
                                  'post_window': f'{pw}D', 'direction': direction,
                                  'threshold': f'{int(thresh*100)}%',
                                  'val_state': vs, 'n_events': len(vs_r)}
                            r2.update(st2); shock_rows.append(r2)

df_shock = pd.DataFrame(shock_rows)
print(f'✅ Shock calibration: {len(df_shock)} rows')
print('\n  AMZN 3D-DOWN >3% by val_state → post 3D:')
s = df_shock[(df_shock['ticker']=='AMZN') & (df_shock['shock_window']=='3D') &
             (df_shock['post_window']=='3D') & (df_shock['direction']=='DOWN') &
             (df_shock['threshold']=='3%')]
print(s[['val_state','n_events','median','p_up','p_dn10']].to_string(index=False))

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 9 — QA CHECKS
# ════════════════════════════════════════════════════════════
qa_rows = []

# Range sanity
for mk, lo, hi, lbl in [
    ('pe_fy1',5,200,'P/E FY1'), ('ev_rev_fy1',0.5,30,'EV/Rev FY1'),
    ('ev_ebitda_fy1',5,80,'EV/EBITDA FY1'), ('ev_fcf_ttm',-500,500,'EV/FCF TTM')]:
    v   = df_piv[mk].dropna()
    oor = v[(v < lo) | (v > hi)]
    pct = (1 - len(oor)/len(v)) * 100
    neg = int((v < 0).sum())
    qa_rows.append(['Range sanity', lbl, f'{pct:.1f}% in [{lo},{hi}]',
                    f'med={v.median():.2f} neg={neg}', 'PASS' if pct >= 90 else 'WARN'])

# Signal direction
for t in CONFIG['TICKERS']:
    d2 = df_piv[df_piv['Symbol']==t][['pe_fy1','log_pe_fy1']].dropna()
    c2 = d2['pe_fy1'].corr(d2['log_pe_fy1'])
    qa_rows.append(['Signal direction', f'{t} P/E vs log P/E',
                    f'corr={c2:.4f}', 'expect >0.95', 'PASS' if c2 > 0.95 else 'WARN'])

# No-backfill proof
for t in CONFIG['TICKERS']:
    df_t  = df_piv[df_piv['Symbol']==t].sort_values('date')
    first = df_t.loc[df_t['eps_est_fy1'].notna(), 'date'].min()
    earlier = int(df_t[df_t['date'] < first]['eps_est_fy1'].notna().sum())
    qa_rows.append(['No-backfill', f'{t} eps_est_fy1',
                    f'first seen={first.strftime("%Y-%m-%d")}',
                    f'backfilled=0', 'PASS' if earlier == 0 else 'FAIL'])

# FCF validity
for t in CONFIG['TICKERS']:
    neg   = int((df_piv[df_piv['Symbol']==t]['fcf_ttm'] < 0).sum())
    total = int(df_piv[df_piv['Symbol']==t]['fcf_ttm'].notna().sum())
    qa_rows.append(['FCF validity', f'{t} fcf_ttm',
                    f'{neg}/{total} negative rows', 'flagged in fcf_flag col', 'INFO'])

# Spot-check 5 AMZN dates
chk = df_piv[df_piv['Symbol']=='AMZN'][['date','pe_fy1','log_pe_fy1']].dropna().sample(5, random_state=42)
chk['manual'] = np.log(chk['pe_fy1'])
chk['match']  = np.isclose(chk['log_pe_fy1'], chk['manual'], atol=1e-6)
for _, r in chk.iterrows():
    qa_rows.append(['Spot-check', f'AMZN {r["date"].strftime("%Y-%m-%d")}',
                    f'P/E={r["pe_fy1"]:.4f}',
                    f'log={r["log_pe_fy1"]:.6f} manual={r["manual"]:.6f}',
                    'PASS' if r['match'] else 'FAIL'])

df_qa = pd.DataFrame(qa_rows, columns=['check_type','series','result','detail','status'])
print(f'✅ QA checks: {len(df_qa)}')
fails = df_qa[df_qa['status']=='FAIL']
if len(fails):
    print(f'⚠️  FAILURES: {len(fails)}')
    print(fails.to_string(index=False))
else:
    print('   All checks PASS or INFO ✓')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 10 — BUILD EXPORT DATAFRAMES
# ════════════════════════════════════════════════════════════

# ── panel_daily export ─────────────────────────────────────
# Rename cols to match Excel exactly
pd_rename = {
    'date':'Date', 'Symbol':'Ticker', 'price':'Price', 'ev':'EV ($M)',
    'eps_est_fy1':'EPS FY1', 'revenue_est_fy1':'Rev FY1 ($M)',
    'ebitda_est_fy1':'EBITDA FY1 ($M)', 'fcf_ttm':'FCF TTM ($M)',
    'capex_ttm':'CapEx TTM ($M)', 'revenue_ttm':'Rev TTM ($M)',
    'fcf_flag':'FCF Flag', 'log_ev_fcf_flag':'log(EV/FCF) Flag',
    'pe_fy1':'P/E FY1', 'ev_rev_fy1':'EV/Rev FY1',
    'ev_ebitda_fy1':'EV/EBITDA FY1', 'ev_fcf_ttm':'EV/FCF TTM',
    'log_pe_fy1':'log P/E', 'log_ev_rev_fy1':'log EV/Rev',
    'log_ev_ebitda_fy1':'log EV/EBITDA', 'log_ev_fcf_ttm':'log EV/FCF',
    'zscore_pe_fy1':'Z P/E FY1', 'pct_pe_fy1':'Pct P/E FY1', 'bucket_pe_fy1':'Bucket P/E FY1',
    'zscore_ev_rev_fy1':'Z EV/Rev FY1', 'pct_ev_rev_fy1':'Pct EV/Rev FY1', 'bucket_ev_rev_fy1':'Bucket EV/Rev FY1',
    'zscore_ev_ebitda_fy1':'Z EV/EBITDA FY1', 'pct_ev_ebitda_fy1':'Pct EV/EBITDA FY1', 'bucket_ev_ebitda_fy1':'Bucket EV/EBITDA FY1',
    'zscore_ev_fcf_ttm':'Z EV/FCF TTM', 'pct_ev_fcf_ttm':'Pct EV/FCF TTM', 'bucket_ev_fcf_ttm':'Bucket EV/FCF TTM',
    'fwd_30d':'Fwd 30d', 'fwd_60d':'Fwd 60d', 'fwd_90d':'Fwd 90d', 'fwd_120d':'Fwd 120d',
}
ordered_cols = list(pd_rename.values())
df_pd_exp = df_piv.rename(columns=pd_rename).copy()
df_pd_exp['Date'] = df_pd_exp['Date'].dt.strftime('%Y-%m-%d')
for col in ['Fwd 30d','Fwd 60d','Fwd 90d','Fwd 120d']:
    if col in df_pd_exp.columns:
        df_pd_exp[col] = (df_pd_exp[col] * 100).round(2)
df_pd_exp = df_pd_exp[[c for c in ordered_cols if c in df_pd_exp.columns]]

# ── val_detail export ─────────────────────────────────────
df_vd_exp = df_val[df_val['bucket_key'].notna()].copy()
df_vd_exp = df_vd_exp[['ticker','metric','bucket_name','horizon_lbl','n',
                        'median','mean','p_up','iqr','p10','p90',
                        'p_dn5','p_dn10','p_dn20','skew','std','stability']]
df_vd_exp.columns = ['ticker','metric','bucket','horizon','n',
                      'median(%)','mean(%)','p_up(%)','iqr(%)',
                      'p10(%)','p90(%)','p_dn5(%)','p_dn10(%)','p_dn20(%)',
                      'skew','std(%)','stability']

# ── val_matrix export — wide pivot ────────────────────────
matrix_rows = []
for ticker in CONFIG['TICKERS']:
    for mk in CONFIG['METRICS']:
        row_out = {'Ticker': ticker, 'Metric': METRIC_LABELS[mk]}
        # build bucket label
        for bkey, bname in zip(CONFIG['BUCKET_KEYS'], CONFIG['BUCKET_NAMES']):
            row_out['Bucket'] = bname
            for h in CONFIG['HORIZONS']:
                sub = df_val[(df_val['ticker']==ticker) & (df_val['metric_key']==mk) &
                             (df_val['bucket_key']==bkey) & (df_val['horizon']==h)]
                if len(sub) == 0:
                    row_out[f'{h}d_med'] = ''; row_out[f'{h}d_pup'] = ''
                    row_out[f'{h}d_pdn10'] = ''; row_out[f'{h}d_n'] = ''
                else:
                    s = sub.iloc[0]
                    row_out[f'{h}d_med']   = s['median']
                    row_out[f'{h}d_pup']   = s['p_up']
                    row_out[f'{h}d_pdn10'] = s['p_dn10']
                    row_out[f'{h}d_n']     = int(s['n'])
            matrix_rows.append(dict(row_out))

df_matrix = pd.DataFrame(matrix_rows)

# ── leaderboard export ────────────────────────────────────
df_l_exp = df_leader[['ticker','metric','horizon','spread_median',
                       'cheap_hit_rate','cheap_n','ic_spearman','rank_quality']].copy()
df_l_exp.columns = ['ticker','metric','horizon','spread_median(%)',
                     'cheap_hit_rate(%)','cheap_n','ic_spearman','rank_quality']

# ── halflife export ───────────────────────────────────────
df_hl_exp = df_hl[['ticker','metric','ar1_coef','halflife_days','speed']].copy()
df_hl_exp.columns = ['ticker','metric','ar1_coefficient','halflife_days','speed_category']

# ── shock_detail export ───────────────────────────────────
df_sd_exp = df_shock[['ticker','shock_window','post_window','direction','threshold',
                       'val_state','n_events','median','mean','p_up','iqr',
                       'p_dn10','p10','p90']].copy()
df_sd_exp.columns = ['ticker','shock_window','post_window','direction','threshold',
                      'val_state','n_events','median(%)','mean(%)','p_up(%)',
                      'iqr(%)','p_dn10(%)','p10(%)','p90(%)']

# ── shock_matrix — at-a-glance pivot ─────────────────────
def fmt_cell(df_shock, ticker, direction, sw, pw, thresh):
    sub = df_shock[(df_shock['ticker']==ticker) & (df_shock['direction']==direction) &
                   (df_shock['shock_window']==f'{sw}D') & (df_shock['post_window']==f'{pw}D') &
                   (df_shock['threshold']==f'{thresh}%') & (df_shock['val_state']=='ALL')]
    if len(sub) == 0: return ''
    s = sub.iloc[0]
    med = s['median(%)']
    pup = round(s['p_up(%)'])
    n   = int(s['n_events'])
    sign = '+' if med >= 0 else ''
    return f'{sign}{med:.1f}% | {pup}%↑ | n={n}'

sm_rows = []
for ticker in CONFIG['TICKERS']:
    for direction in ['DOWN','UP']:
        row = {'Ticker': ticker, 'Direction': direction}
        for sw in CONFIG['SHOCK_WINDOWS']:
            for thresh_pct in [3, 5]:
                for pw in CONFIG['POST_WINDOWS']:
                    col = f'Shock {sw}D→{pw}D\n({thresh_pct}%)'
                    row[col] = fmt_cell(df_sd_exp, ticker, direction, sw, pw, thresh_pct)
        sm_rows.append(row)

df_sm = pd.DataFrame(sm_rows)

print('✅ All export dataframes built')
print(f'   panel_daily : {len(df_pd_exp):,} rows × {len(df_pd_exp.columns)} cols')
print(f'   val_detail  : {len(df_vd_exp)} rows')
print(f'   val_matrix  : {len(df_matrix)} rows')
print(f'   leaderboard : {len(df_l_exp)} rows')
print(f'   halflife    : {len(df_hl_exp)} rows')
print(f'   shock_detail: {len(df_sd_exp)} rows')
print(f'   shock_matrix: {len(df_sm)} rows')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 11 — BUILD current_signals
# ════════════════════════════════════════════════════════════
latest_date = df_piv['date'].max()
df_now = df_piv[df_piv['date'] == latest_date].copy()

BUCKET_LABEL = {
    'D1_cheap':'D1 Cheap (bot 10%)', 'Q2_cheap_mid':'Q2 Cheap-Mid (10-25%)',
    'Q3_neutral':'Q3 Neutral (25-75%)', 'Q4_rich_mid':'Q4 Rich-Mid (75-90%)',
    'D5_expensive':'D5 Expensive (top 10%)', '':''
}

cs_rows = []
for ticker in CONFIG['TICKERS']:
    row_now = df_now[df_now['Symbol'] == ticker]
    if len(row_now) == 0: continue
    r = row_now.iloc[0]
    for mk in CONFIG['METRICS']:
        ratio   = r.get(mk)
        log_val = r.get(f'log_{mk}')
        z_val   = r.get(f'zscore_{mk}')
        pct_val = r.get(f'pct_{mk}')
        bkt_raw = r.get(f'bucket_{mk}', '')
        bkt_lbl = BUCKET_LABEL.get(str(bkt_raw), '')
        fcf_flag     = r.get('fcf_flag','') if mk == 'ev_fcf_ttm' else ''
        log_fcf_flag = r.get('log_ev_fcf_flag','') if mk == 'ev_fcf_ttm' else ''

        # Signal
        if bkt_raw in ('D1_cheap','Q2_cheap_mid'):
            sig = 'CHEAP ↑'
        elif bkt_raw in ('D5_expensive','Q4_rich_mid'):
            sig = 'RICH ↓'
        elif pd.notna(ratio) and ratio < 0:
            sig = 'NEUTRAL'
        else:
            sig = 'NEUTRAL' if pd.notna(ratio) else ''

        # Override: negative FCF → NEUTRAL
        if mk == 'ev_fcf_ttm' and str(fcf_flag) == 'negative_fcf':
            sig = 'NEUTRAL'; bkt_lbl = ''

        is_best = 'YES' if best_anchor.get(ticker) == METRIC_LABELS[mk] else ''

        cs_rows.append({
            'Ticker':          ticker if mk == CONFIG['METRICS'][0] else '',
            'Metric':          METRIC_LABELS[mk],
            'Ratio Value':     round(float(ratio), 2) if pd.notna(ratio) else '',
            'log(Ratio)':      round(float(log_val), 4) if pd.notna(log_val) else '',
            'Z-Score (252d)':  round(float(z_val), 3) if pd.notna(z_val) else '',
            'Pct Rank (%)':    round(float(pct_val)*100, 1) if pd.notna(pct_val) else '',
            'Bucket':          bkt_lbl,
            'Signal':          sig,
            'FCF Flag':        fcf_flag,
            'log EV/FCF Flag': log_fcf_flag,
            'Best Anchor?':    is_best,
        })

df_cs = pd.DataFrame(cs_rows)
print(f'✅ current_signals: {len(df_cs)} rows')
print(f'   Snapshot date: {latest_date.strftime("%Y-%m-%d")}')
print(df_cs[['Ticker','Metric','Ratio Value','Pct Rank (%)','Signal','Best Anchor?']].to_string(index=False))

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 12 — WRITE EXCEL  (9 tabs, no README)
# ════════════════════════════════════════════════════════════
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = Workbook()
wb.remove(wb.active)  # remove default sheet

HDR_BLUE   = PatternFill('solid', fgColor='1F4E79')
HDR_GREEN  = PatternFill('solid', fgColor='375623')
HDR_ORANGE = PatternFill('solid', fgColor='833C00')
ALT_BLUE   = PatternFill('solid', fgColor='EBF3FB')
ALT_GREEN  = PatternFill('solid', fgColor='EBF5EB')
WHITE      = PatternFill('solid', fgColor='FFFFFF')
GREY       = PatternFill('solid', fgColor='F2F2F2')
thin = Side(style='thin', color='CCCCCC')
bdr  = Border(left=thin, right=thin, top=thin, bottom=thin)

def hdr_font(color='FFFFFF'): return Font(bold=True, color=color, name='Arial', size=9)
def body_font(bold=False, color='000000'): return Font(bold=bold, color=color, name='Arial', size=9)
def ctr(): return Alignment(horizontal='center', vertical='center', wrap_text=False)
def lft(): return Alignment(horizontal='left', vertical='center', wrap_text=False)

def write_df(ws, df, title, title_fill=HDR_BLUE, hdr_fill=HDR_BLUE,
             alt_fill=ALT_BLUE, freeze='A3'):
    """Write title row, header row, then data rows to worksheet."""
    # Title
    ws.merge_cells(f'A1:{get_column_letter(len(df.columns))}1')
    c = ws.cell(1, 1, title)
    c.fill = title_fill; c.font = hdr_font(); c.alignment = lft()
    ws.row_dimensions[1].height = 18

    # Headers
    for ci, col in enumerate(df.columns, 1):
        c = ws.cell(2, ci, col)
        c.fill = hdr_fill; c.font = hdr_font(); c.alignment = ctr(); c.border = bdr
    ws.row_dimensions[2].height = 20

    # Data
    for ri, (_, row) in enumerate(df.iterrows(), 3):
        bg = alt_fill if ri % 2 == 0 else WHITE
        for ci, val in enumerate(row, 1):
            c = ws.cell(ri, ci, val if val != '' else None)
            c.fill = bg; c.font = body_font(); c.alignment = ctr() if ci > 1 else lft()
            c.border = bdr
        ws.row_dimensions[ri].height = 16

    if freeze:
        ws.freeze_panes = freeze
    ws.sheet_view.showGridLines = False

# ── 1. panel_daily ─────────────────────────────────────────
ws = wb.create_sheet('panel_daily')
ws.cell(1,1,'panel_daily').fill=HDR_BLUE
ws.cell(1,1).font = hdr_font()
# Section row 2
sections = [('Raw Data',1,10),('Flags',11,12),('Ratios',13,16),
            ('Log Multiples',17,20),('Signals',21,32),('Fwd Returns',33,36)]
for lbl,c1,c2 in sections:
    if c1==c2: ws.cell(2,c1,lbl)
    else: ws.merge_cells(f'{get_column_letter(c1)}2:{get_column_letter(c2)}2')
    ws.cell(2,c1,lbl).fill = HDR_BLUE
    ws.cell(2,c1).font = hdr_font()
    ws.cell(2,c1).alignment = ctr()
ws.row_dimensions[2].height = 16
# Headers row 3
for ci,col in enumerate(df_pd_exp.columns,1):
    c = ws.cell(3,ci,col)
    c.fill=HDR_BLUE; c.font=hdr_font(); c.alignment=ctr(); c.border=bdr
ws.row_dimensions[3].height=20
# Data
for ri,(_, row) in enumerate(df_pd_exp.iterrows(),4):
    bg = ALT_BLUE if ri%2==0 else WHITE
    for ci,val in enumerate(row,1):
        c=ws.cell(ri,ci,val if val!='' else None)
        c.fill=bg; c.font=body_font(); c.alignment=ctr() if ci>1 else lft(); c.border=bdr
ws.freeze_panes='A4'
ws.sheet_view.showGridLines=False
for ci,w in enumerate([12,8,10,14,10,14,14,12,12,12,12,16,10,12,14,12,
                        10,12,14,12,12,12,18,12,12,18,14,14,20,12,12,18,9,9,9,9],1):
    ws.column_dimensions[get_column_letter(ci)].width=w
print('  ✓ panel_daily written')

# ── 2. current_signals ─────────────────────────────────────
ws = wb.create_sheet('current_signals')
ws.merge_cells('A1:K1')
ws.cell(1,1,'Current Valuation Snapshot — As of ' + latest_date.strftime('%Y-%m-%d')).fill=HDR_BLUE
ws.cell(1,1).font=hdr_font(); ws.cell(1,1).alignment=lft()
ws.merge_cells('A2:K2')
ws.cell(2,1,'Identity').fill=HDR_BLUE; ws.cell(2,1).font=hdr_font()
cs_hdrs = list(df_cs.columns)
for ci,h in enumerate(cs_hdrs,1):
    c=ws.cell(3,ci,h); c.fill=HDR_BLUE; c.font=hdr_font(); c.alignment=ctr(); c.border=bdr
ws.row_dimensions[3].height=20
SIGNAL_FILLS = {'CHEAP ↑':PatternFill('solid',fgColor='C6EFCE'),
                'RICH ↓':PatternFill('solid',fgColor='FFC7CE'),
                'NEUTRAL':PatternFill('solid',fgColor='F2F2F2')}
ri=4
prev_ticker=None
for _,row in df_cs.iterrows():
    if row['Ticker'] and row['Ticker']!=prev_ticker and prev_ticker is not None:
        # spacer
        for ci in range(1,12): ws.cell(ri,ci,'').fill=GREY
        ws.row_dimensions[ri].height=4; ri+=1
    prev_ticker=row['Ticker'] if row['Ticker'] else prev_ticker
    sig = str(row.get('Signal',''))
    bg = SIGNAL_FILLS.get(sig, WHITE)
    for ci,val in enumerate(row,1):
        c=ws.cell(ri,ci,val if val!='' else None)
        c.fill=bg if ci==8 else WHITE; c.font=body_font(); c.alignment=ctr() if ci>1 else lft(); c.border=bdr
    ws.row_dimensions[ri].height=18; ri+=1
ws.freeze_panes='A4'
ws.sheet_view.showGridLines=False
for ci,w in enumerate([9,17,12,12,14,13,23,12,13,17,12],1):
    ws.column_dimensions[get_column_letter(ci)].width=w
print('  ✓ current_signals written')

# ── 3. val_matrix ──────────────────────────────────────────
ws = wb.create_sheet('val_matrix')
ws.merge_cells('A1:S1')
ws.cell(1,1,'Valuation Matrix Dashboard').fill=HDR_BLUE; ws.cell(1,1).font=hdr_font()
# Row 2: span headers
ws.cell(2,1,'Ticker'); ws.cell(2,2,'Metric'); ws.cell(2,3,'Bucket')
for i,h in enumerate(CONFIG['HORIZONS']):
    base=4+i*4
    ws.merge_cells(f'{get_column_letter(base)}2:{get_column_letter(base+3)}2')
    ws.cell(2,base,f'{h}D').fill=HDR_BLUE; ws.cell(2,base).font=hdr_font(); ws.cell(2,base).alignment=ctr()
for ci in [1,2,3]:
    ws.cell(2,ci).fill=HDR_BLUE; ws.cell(2,ci).font=hdr_font()
# Row 3: sub-headers
sub_hdrs=['Ticker','Metric','Bucket']
for h in CONFIG['HORIZONS']:
    sub_hdrs+=[f'Med%',f'P(Up)%',f'P(<-10%)%',f'n']
for ci,h in enumerate(sub_hdrs,1):
    c=ws.cell(3,ci,h); c.fill=HDR_BLUE; c.font=hdr_font(); c.alignment=ctr(); c.border=bdr
ws.row_dimensions[3].height=20
ri=4
for _,row in df_matrix.iterrows():
    bg=ALT_BLUE if ri%2==0 else WHITE
    vals=[row['Ticker'],row['Metric'],row['Bucket']]
    for h in CONFIG['HORIZONS']:
        vals+=[row[f'{h}d_med'],row[f'{h}d_pup'],row[f'{h}d_pdn10'],row[f'{h}d_n']]
    for ci,v in enumerate(vals,1):
        c=ws.cell(ri,ci,v if v!='' else None)
        c.fill=bg; c.font=body_font(); c.alignment=ctr() if ci>2 else lft(); c.border=bdr
    ws.row_dimensions[ri].height=16; ri+=1
ws.freeze_panes='A4'; ws.sheet_view.showGridLines=False
for ci,w in enumerate([8,16,24]+[8,8,10,6]*4,1):
    ws.column_dimensions[get_column_letter(ci)].width=w
print('  ✓ val_matrix written')

# ── 4–9. Remaining tabs ────────────────────────────────────
tab_specs = [
    ('val_detail',   df_vd_exp,  'Valuation Bucket Detail — All 17 stats per ticker × metric × bucket × horizon', HDR_BLUE),
    ('leaderboard',  df_l_exp,   'Metric Leaderboard — Valuation anchor ranking',    HDR_GREEN),
    ('halflife',     df_hl_exp,  'AR(1) Half-Life — How fast valuation gaps close',  HDR_BLUE),
    ('shock_matrix', df_sm,      'Shock Matrix — At-a-glance: Median | P(up)% | n',  HDR_ORANGE),
    ('shock_detail', df_sd_exp,  'Shock Calibration Detail — ALL + val-state conditioning', HDR_ORANGE),
    ('qa_checks',    df_qa,      'QA / Sanity Checks',                               HDR_GREEN),
]
for tab_name, df_tab, title, hdr_col in tab_specs:
    ws = wb.create_sheet(tab_name)
    write_df(ws, df_tab, title, title_fill=hdr_col, hdr_fill=hdr_col)
    # Auto-width
    for ci in range(1, len(df_tab.columns)+1):
        ws.column_dimensions[get_column_letter(ci)].width = 16
    print(f'  ✓ {tab_name} written ({len(df_tab)} rows)')

wb.save(CONFIG['OUTPUT_EXCEL'])
print(f'\n✅ Excel saved: {CONFIG["OUTPUT_EXCEL"]}')
print(f'   Tabs: {wb.sheetnames}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 13 — DOWNLOAD EXCEL
# ════════════════════════════════════════════════════════════
files.download(CONFIG['OUTPUT_EXCEL'])
print(f'✅ Download started: {CONFIG["OUTPUT_EXCEL"]}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 14 — AUTHENTICATE & WRITE TO GOOGLE SHEETS
# ════════════════════════════════════════════════════════════
print('Authenticating with Google...')
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
print('✅ Authenticated')

sh = gc.open_by_url(CONFIG['SHEET_URL'])
print(f'✅ Opened: {sh.title}')

def upsert_sheet(sh, name, df, freeze_rows=1):
    """Create or clear-and-rewrite a worksheet tab."""
    try:
        ws = sh.worksheet(name)
        ws.clear()
    except gspread.exceptions.WorksheetNotFound:
        ws = sh.add_worksheet(title=name, rows=max(len(df)+5,100), cols=max(len(df.columns)+2,26))
    df_clean = df.copy()
    for col in df_clean.columns:
        if pd.api.types.is_datetime64_any_dtype(df_clean[col]):
            df_clean[col] = df_clean[col].dt.strftime('%Y-%m-%d')
        elif df_clean[col].dtype == object:
            df_clean[col] = df_clean[col].astype(str).replace('nan','')
    set_with_dataframe(ws, df_clean, include_index=False)
    if freeze_rows:
        ws.freeze(rows=freeze_rows)
    print(f'  ✓ {name}: {len(df):,} rows written')

print('\nWriting all tabs to Google Sheets...')
upsert_sheet(sh, 'panel_daily',    df_pd_exp)
upsert_sheet(sh, 'current_signals',df_cs)
upsert_sheet(sh, 'val_matrix',     df_matrix)
upsert_sheet(sh, 'val_detail',     df_vd_exp)
upsert_sheet(sh, 'leaderboard',    df_l_exp)
upsert_sheet(sh, 'halflife',       df_hl_exp)
upsert_sheet(sh, 'shock_matrix',   df_sm)
upsert_sheet(sh, 'shock_detail',   df_sd_exp)
upsert_sheet(sh, 'qa_checks',      df_qa)

print(f'\n🎉 All 9 tabs written to Google Sheet: {sh.url}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 15 — SNAPSHOT: Current signals summary print
# ════════════════════════════════════════════════════════════
print(f'=== Current Valuation Snapshot ({latest_date.strftime("%Y-%m-%d")}) ===')
for ticker in CONFIG['TICKERS']:
    row = df_now[df_now['Symbol'] == ticker]
    if len(row) == 0: continue
    row = row.iloc[0]
    print(f'\n{ticker}:')
    for mk in CONFIG['METRICS']:
        ratio = row.get(mk)
        pct   = row.get(f'pct_{mk}')
        bkt   = row.get(f'bucket_{mk}','')
        z     = row.get(f'zscore_{mk}')
        if pd.notna(ratio):
            print(f'  {METRIC_LABELS[mk]:18s}  {ratio:7.2f}x   z={z:+.2f}  pct={pct*100:.0f}th   [{bkt}]')

print('\n=== Best Anchor (60d, by rank_quality) ===')
top60 = (df_leader[df_leader['horizon']=='60d']
         .sort_values(['ticker','rank_quality'], ascending=[True,False])
         .groupby('ticker').first().reset_index())
print(top60[['ticker','metric','spread_median','cheap_hit_rate','ic_spearman']].to_string(index=False))

print('\n=== Half-Life Summary ===')
print(df_hl[['ticker','metric','halflife_days','speed']].to_string(index=False))